# DiagnosticIQ source step 4: rule condition trees

Parses each rule's condition text into a condition tree and pickles the rule set to `extracted/TreeStruct.pkl`, the input to `diagnosticiq_simple_dataset_creation.ipynb`.

`Node` is defined inline here, so the pickle refers to `__main__.Node`. `dataset_utils.utils.file_handle.load_pickle` maps it to `dataset_utils.utils.tree.Node` when loading.

In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any
from abc import abstractmethod, ABC
import pickle
import copy

In [ ]:
#parameter 

from dataset_utils.paths import get_dataset_dir

extract_fol = get_dataset_dir() / "extracted"
dataset_loc = os.path.join(extract_fol,"observations.rules.json")

In [ ]:
def save_json(data:dict, loc:str) -> None:
    with open(loc, "w") as f0:
        json.dump(data, f0)


def text_write(text, filename= "debug.test"):
    with open(filename, "w") as f0:
        f0.write(text)

def dict_value_string_function(d:dict, func:Callable[[Optional[Any]], Optional[Any]]) -> dict:
    r_dict = {}
    for k,v in d.items():
        if isinstance(v, dict):
            r_dict[k] = dict_value_string_function(v,func)
        else:
            r_dict[k] = func(v)

    return r_dict

'''
class ops:
    _OPS = None
    def __init__(self, statement) -> None:
        self.subject = None
        self.value = None
        ops._process_statement(statement)

    @abstractmethod
    def _create_subject_value(self, statement:str):
        raise NotImplementedError

    @staticmethod
    def _process_statement(statement:str):

        pass
        
class OR(ops):
    _OPS = "OR"
    def __init__(self, statement) -> None:
        super().__init__(statement)

    def _create_subject_value(self, statement: str):
        pass
'''

class OPS:
    ops = ["OR", "AND"]

    def __init__(self) -> None:
        for o in self.ops:
            setattr(self,o,o)


    
'''
def print_tree(root, level=0, prefix="Root: "):
    if root is not None:
        print_tree(root.right, level+1, "R---- ")
        print("    "*level + prefix + root.val)
        print_tree(root.left, level+1, "L---- ")


class Node:
    def __init__(self, ops:str, statement:str) -> None:
        self.left = None
        self.val = None
        self.right = None
        self.ops = ops
        self.statement = statement
        self.build_graph()

    def build_graph(self):
        split = self.statement.split(self.ops)
        if len(split)>=2:
            self.left = Node(self.ops,split[0])
            self.val = self.ops
            self.right = Node(self.ops, self.ops.join(split[1:]))
        else:
            self.val = self.statement


    @staticmethod
    def generate_graph(statement:str) -> Node:
        root = None

        for oi, o in enumerate(OPS.ops):
            if oi == 0:
                root = Node(o,statement)

                return root
            
'''

def print_tree(root, level=0, prefix="Root: "):
    if root is not None:
        print("    "*level + prefix + root.val)
        for x in root.children:
            print_tree(x, level+1, "L---- ")


class Node:
    def __init__(self, ops:str, statement:Optional[str], manual=False) -> None:
        self.children = []
        self.val = None
        self.ops = ops
        self.statement = statement

        if not manual:
            self.build_graph()

    def build_graph(self):
        split = self.statement.split(self.ops)
        if len(split)>=2:
            self.children = [Node(self.ops,split[0]), Node(self.ops, self.ops.join(split[1:]))]
            self.val = self.ops
        else:
            self.val = self.statement


    @staticmethod
    def generate_graph(statement:str) -> "Node":
        root = None

        for oi, o in enumerate(OPS.ops):
            if oi == 0:
                root = Node(o,statement)

                return root


def common_string_replace(string:str):
    return string.replace("\n", " ").replace("AND/OR", "OR")


def process_rule(rule_str:str) -> Node:
    # break rule list
    individual_rules = list(map(lambda x: x.strip(),rule_str.split("\n -")))
    individual_rules[0] = individual_rules[0].replace("- ", "")

    #print(individual_rules)

    #extract conditionals
    cond_info_rules = []
    for i in range(1,len(individual_rules)):
        r1 = common_string_replace(individual_rules[i-1])
        r2 = common_string_replace(individual_rules[i])

        #print(r1,"$$$")
        #print(r2, "$$$$")

        try:
            _cond_r1 = "OR" in r1[:4]
        except IndexError:
            _cond_r1 = False

        try:
            _cond_r2 = "OR" in r2[:4]
        except IndexError:
            _cond_r2 = False
            

        if  _cond_r1 and _cond_r2:
            cond_info_rules[-1].append(r1[2:].strip())
        elif _cond_r2:
            #cond_info_rules.append([r1,r2[2:].strip()])
            cond_info_rules.append([r1])
        elif _cond_r1:
            cond_info_rules[-1].append(r1[2:].strip())
        else:
            cond_info_rules.append(r1)

        if i == len(individual_rules) - 1 :
            if _cond_r2:
                cond_info_rules[-1].append(r2[2:].strip())
            else:
                cond_info_rules.append(r2)

        #print(cond_info_rules, "****")

    
    full_root = Node('AND', rule_str, True)
    full_root.children = []
    full_root.val = "root"
    for c in cond_info_rules:
        if isinstance(c,str):
            full_root.children.append(Node.generate_graph(c))
        else:
            _temp = Node('OR', None, True)
            _temp.children = [Node.generate_graph(x) for x in c]
            _s = " OR ".join([x.statement for x in _temp.children])
            _temp.statement = _s
            _temp.val = 'OR'
            full_root.children.append(_temp)

    return full_root


def save_pickle(obj:Any, loc:str)->None:
    with open(loc, "wb") as f0:
        pickle.dump(obj,f0)

def load_pickle(loc:str) -> Any:
    with open(loc, "rb") as f0:
        return pickle.load(f0)


In [ ]:
with open(dataset_loc, "r") as f0:
    data = json.load(f0)

processed_data = copy.deepcopy(data)
for pd, d in zip(processed_data["rule_set"],data["rule_set"]):
    pd["display_text"]["rules"] = process_rule(d["display_text"]["rules"])

In [ ]:
for i,pd in enumerate(processed_data["rule_set"]):
    print(f"Asset:{pd['asset']}")
    print("Rules:")
    print_tree(pd["display_text"]["rules"])
    print("Observations:")
    print(pd["display_text"]["observations"])
    print("\n")
    

In [ ]:
save_pickle(
    processed_data,
    os.path.join(extract_fol, "TreeStruct.pkl")
)